# Transfer learning on a flower photo dataset

**Ernestas Lazurkevicius - PRIf-24/2, VilniusTech**

An adaptation of the lecturer's `cnn_Transfer_learning.ipynb` (Algirdas Laukaitis).

## What the original notebook does

The original demonstrates **transfer learning**: instead of training an image
classifier from nothing, it takes a convolutional network already trained on millions
of ImageNet photos, freezes it, and trains only a small new output layer on top.

* **Chapter 1** loads CIFAR-10 (50,000 32x32 images, 10 classes), wraps the TF-Hub
  `efficientnet_v2_imagenet21k_b1` feature vector in a frozen `hub.KerasLayer`, adds
  `Dense(10)` and trains for 3 epochs. Only **12,810** of 6,943,934 parameters are
  trainable - 0.18 % of the model. It reaches **88.5 %** test accuracy and saves the
  model.
* **Chapter 2** reloads that model, unfreezes everything, divides the learning rate by
  10, applies data augmentation and trains further.

The lesson is the economics: 12,810 trainable parameters and three epochs buy 88.5 %
accuracy, because the frozen layers already know edges, textures and shapes. Training
an equivalent network from scratch would take far longer and far more data.

## What this notebook changes

The method is preserved - frozen pretrained feature extractor, new classification head,
then a fine-tuning stage with augmentation. The data changes, and two pieces of
machinery are modernised for reasons explained where they occur.

| | Original | This notebook |
|---|---|---|
| Dataset | CIFAR-10, 50,000 images at 32x32 | **tf_flowers**, 3,670 photos at full resolution |
| Classes | 10 everyday objects | 5 flower species |
| Feature extractor | TF-Hub `efficientnet_v2_..._b1/feature_vector` | `keras.applications.EfficientNetV2B1` |
| Input pipeline | NumPy arrays held in memory | `tf.data` pipeline |

**Why tf_flowers.** CIFAR-10 is a poor match for this technique, and the original's own
numbers show it. Its images are 32x32, but the feature extractor expects 240x240 and was
trained on real photographs. Upscaling a 32x32 thumbnail to 240x240 produces a blurry
image with no fine texture, so most of what the pretrained layers know is wasted. That is
why the result stalls at 88.5 %. tf_flowers holds real photographs at natural resolution,
which is what a pretrained photo model is built for - and with only 3,670 images, it is
exactly the situation where transfer learning is the right tool rather than a shortcut.

> **Using your own photos instead.** Only the loading cell in Step 1 is dataset-specific.
> Replacing it with `tf.keras.utils.image_dataset_from_directory("my_photos")` over a
> folder-per-class layout makes everything below work unchanged.

Every deviation from the original is marked with a **`# CHANGE:`** comment.

> **Status: not yet executed.** Unlike my tabular notebook, this one could not be run
> before delivery - the environment it was written in cannot reach TF Hub, Kaggle or the
> dataset mirrors. Every cell has been syntax-checked and reviewed, but the outputs below
> appear only after the first Colab run. Any accuracy figure quoted in the text is taken
> from the **original notebook's** saved output, never invented for this one.

---
# Chapter 1. Build the model, train it, and save it

## Step 1. Download and prepare the flower dataset

`tensorflow_datasets` downloads and caches the dataset, and `with_info=True` returns the
class names so they never have to be hard-coded (the original hard-codes a
`class_names` list, which silently breaks if the label order ever changes).

One structural difference matters: CIFAR-10 ships with an official train/test split, but
**tf_flowers has only a single `train` split**. It must be divided manually, which is
what the slice syntax does - 80 % train, 10 % validation, 10 % test.

In [ ]:
# tensorflow-datasets does not declare importlib_resources as a dependency, but
# tensorflow_datasets/core/dataset_builder.py imports it. On Python 3.13 that
# surfaces as "ModuleNotFoundError: No module named 'importlib_resources'" the first
# time tfds.load() tries to prepare a dataset, so it is installed explicitly here.
%pip install -q tensorflow-datasets importlib_resources

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

print("TensorFlow", tf.__version__)

In [ ]:
# CHANGE: tf_flowers via tensorflow_datasets, replacing the CIFAR-10 loader.
# tf_flowers has no official test split, so it is sliced manually here.
(train_raw, validation_raw, test_raw), info = tfds.load(
    "tf_flowers",
    split=["train[:80%]", "train[80%:90%]", "train[90%:]"],
    as_supervised=True,     # yields (image, label) pairs instead of a dict
    with_info=True,
)

# CHANGE: class names come from the dataset metadata, not a hard-coded list.
class_names = info.features["label"].names
num_classes = info.features["label"].num_classes

print("Classes:", class_names)
print("Train / validation / test:",
      train_raw.cardinality().numpy(),
      validation_raw.cardinality().numpy(),
      test_raw.cardinality().numpy())

### If `tensorflow_datasets` gives trouble

`tfds` pulls in a fair number of dependencies and is the most fragile part of this
notebook. The cell below loads the **same** flower photos without it, by downloading the
folder-per-class archive directly and reading it with
`image_dataset_from_directory`.

It is worth knowing about for a second reason: that folder-per-class layout is exactly
how your own photos would be organised, so this cell is also the "use my own data" path.
Point `data_dir` at a folder of your own and nothing below changes.

In [ ]:
# ALTERNATIVE LOADER - no tensorflow_datasets required.
# Uncomment this whole cell to use it instead of the tfds cell above.
#
# import pathlib
#
# archive = tf.keras.utils.get_file(
#     "flower_photos.tgz",
#     origin="https://storage.googleapis.com/download.tensorflow.org/"
#            "example_images/flower_photos.tgz",
#     extract=True,
# )
#
# # get_file returns the archive path on some Keras versions and the extracted folder
# # on others, so locate the folder explicitly rather than assuming.
# data_dir = next(p for p in pathlib.Path(archive).parent.rglob("flower_photos")
#                 if p.is_dir())
# print("Data directory:", data_dir)
#
# # To use YOUR OWN photos instead, replace the three lines above with:
# #   data_dir = "/content/my_photos"      # one subfolder per class
#
# full_ds = tf.keras.utils.image_dataset_from_directory(
#     data_dir,
#     image_size=(240, 240),   # IMAGE_SIZE is defined in Step 2, below this cell
#     batch_size=None,         # unbatched, so prepare() in Step 2 can batch it
#     shuffle=True,
#     seed=42,
#     label_mode="int",
# )
#
# class_names = full_ds.class_names
# num_classes = len(class_names)
#
# count = int(full_ds.cardinality().numpy())
# n_train, n_validation = int(0.8 * count), int(0.1 * count)
#
# train_raw = full_ds.take(n_train)
# remainder = full_ds.skip(n_train)
# validation_raw = remainder.take(n_validation)
# test_raw = remainder.skip(n_validation)
#
# print("Classes:", class_names)
# print("Train / validation / test:",
#       n_train, n_validation, count - n_train - n_validation)

### Look at the data before modelling anything

The original never displays its input images until after training. Looking first is
cheap and catches whole categories of mistake - wrong labels, wrong colour channels,
images that are not what you expected.

In [ ]:
# CHANGE (new cell): inspect the raw data before building anything.
plt.figure(figsize=(10, 6))
for i, (image, label) in enumerate(train_raw.take(8)):
    plt.subplot(2, 4, i + 1)
    plt.imshow(image.numpy().astype("uint8"))
    plt.title(class_names[label.numpy()], fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Step 2. The input pipeline

The original holds the whole dataset in memory as NumPy arrays. That works for CIFAR-10
because 32x32 images are tiny, but it does not generalise: tf_flowers photos vary in
size, so they cannot be stacked into one array until they are resized.

A `tf.data` pipeline is the normal answer. It resizes on the fly, batches, and prefetches
the next batch while the current one trains.

### The one detail that silently destroys accuracy

The original divides its images by 255 to put them in the `[0, 1]` range, because
**TF-Hub image modules expect `[0, 1]` input**.

`keras.applications.EfficientNetV2B1` is the opposite: it expects **raw `[0, 255]`
values**, because it carries its own rescaling layer inside the model. Feeding it `[0, 1]`
data does not raise an error - it trains, and it simply performs much worse, because
every input is 255x darker than the network expects.

So the division by 255 is deliberately **absent** below. This is the single most
important line-level difference between the two notebooks.

In [ ]:
IMAGE_SIZE = (240, 240)   # the resolution EfficientNetV2B1 was trained at
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def prepare(dataset, shuffle=False):
    # CHANGE: resize only. NO division by 255 - keras.applications EfficientNetV2
    # rescales internally and expects inputs in the [0, 255] range.
    dataset = dataset.map(
        lambda image, label: (tf.image.resize(image, IMAGE_SIZE), label),
        num_parallel_calls=AUTOTUNE,
    )
    if shuffle:
        dataset = dataset.shuffle(1000)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = prepare(train_raw, shuffle=True)
validation_ds = prepare(validation_raw)
test_ds = prepare(test_raw)

images, labels = next(iter(train_ds))
print("Batch shape:", images.shape)
print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

## Step 3. The model

Three pieces, exactly as in the original:

1. a **frozen** pretrained convolutional base that turns an image into a feature vector,
2. global average pooling (`pooling="avg"`) to collapse the feature map to one vector,
3. a single `Dense(num_classes)` layer producing logits.

`base.trainable = False` is what makes this transfer learning rather than ordinary
training: the pretrained weights are held fixed and only the final layer learns.

### Why `keras.applications` instead of TF Hub

The original needs three awkward lines - `!pip install tf_keras`, `import tf_keras`, and
building the model with `tf_keras.Sequential` instead of `tf.keras.Sequential`. The
reason is a version conflict: `tensorflow_hub.KerasLayer` was written against Keras 2,
and modern TensorFlow ships Keras 3, which is not backwards-compatible. `tf_keras` is a
compatibility package that provides the old Keras 2 API alongside the new one.

That shim works but it is fragile, and it means the notebook runs on a deprecated API.
`keras.applications.EfficientNetV2B1` is the same architecture with the same ImageNet
weights, maintained inside Keras itself, with no shim. The TF-Hub version is kept as a
commented alternative directly below, so either can be defended.

In [ ]:
# CHANGE: keras.applications instead of hub.KerasLayer + the tf_keras shim.
base_model = tf.keras.applications.EfficientNetV2B1(
    include_top=False,              # drop ImageNet's own 1000-class classifier
    weights="imagenet",
    input_shape=IMAGE_SIZE + (3,),
    pooling="avg",                  # feature map -> one vector per image
)

base_model.trainable = False        # THIS is what makes it transfer learning

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
features = base_model(inputs, training=False)   # keeps BatchNorm in inference mode
outputs = tf.keras.layers.Dense(num_classes)(features)   # logits, no activation

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# The original's TF-Hub route, kept for reference. To use it instead, install
# tensorflow_hub and tf_keras, then build the model with tf_keras rather than tf.keras:
#
# !pip install -q tensorflow_hub tf_keras
# import tensorflow_hub as hub, tf_keras
# feature_extractor_layer = hub.KerasLayer(
#     "https://tfhub.dev/google/imagenet/efficientnet_v2_imagenet21k_b1/feature_vector/2",
#     input_shape=IMAGE_SIZE + (3,), trainable=False)
# model = tf_keras.Sequential([
#     tf_keras.layers.Resizing(*IMAGE_SIZE),
#     feature_extractor_layer,
#     tf_keras.layers.Dense(num_classes)])
#
# Note: the TF-Hub module expects [0, 1] input, so the pipeline above would then need
# its division by 255 restored.

### How little is actually being trained

`model.summary()` prints the totals, but the ratio is the point of the whole technique,
so it is worth stating explicitly. In the original this was 12,810 trainable parameters
out of 6,943,934 - about 0.18 %.

In [ ]:
# CHANGE (new cell): make the transfer-learning ratio explicit.
trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
frozen = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))

print(f"Trainable parameters:     {trainable:,}")
print(f"Frozen parameters:        {frozen:,}")
print(f"Share being trained:      {100 * trainable / (trainable + frozen):.2f} %")

## Step 4. Train the head

Three epochs, as in the original. Each epoch is slow for the number of parameters
involved, because every image still has to pass forward through all 6.9 million frozen
weights - only the backward pass is cheap.

In [ ]:
history = model.fit(train_ds, validation_data=validation_ds, epochs=3)

In [ ]:
# CHANGE (new cell): plot the learning curves.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(history.history["loss"], marker="o", label="training")
ax1.plot(history.history["val_loss"], marker="o", label="validation")
ax1.set_title("Loss"); ax1.set_xlabel("epoch"); ax1.legend()

ax2.plot(history.history["accuracy"], marker="o", label="training")
ax2.plot(history.history["val_accuracy"], marker="o", label="validation")
ax2.set_title("Accuracy"); ax2.set_xlabel("epoch"); ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print(f"Test accuracy after Chapter 1: {test_accuracy:.4f}")

# CHANGE (new cell): record the figure so Chapter 2 can be judged against it.
frozen_stage_accuracy = float(test_accuracy)

## Step 5. Save the model

The original saves to the TensorFlow SavedModel directory format and then zips it, which
is why Chapter 2 has to `unzip` before loading. The modern single-file `.keras` format
removes both steps.

In [ ]:
# CHANGE: single-file .keras format instead of a SavedModel directory plus zip.
export_path = "flowers_transfer_model.keras"
model.save(export_path)
print("Saved to", export_path)

## Look at the predictions

A grid of test images labelled `true / predicted`, green when correct and red when wrong,
exactly as the original does. Looking at the mistakes is more informative than the
accuracy number: confusions between visually similar species are expected, while a
confusion between, say, dandelions and roses would suggest something is wrong.

The original plots 100 CIFAR-10 thumbnails. These photos are much larger, so 25 are
shown at a readable size instead.

In [ ]:
# CHANGE: 25 images at a readable size, and predictions computed in ONE batched
# call. The original calls model.predict() separately for each of 100 images inside a
# Python loop, which is far slower than predicting on a batch.
images, labels = next(iter(test_ds))
logits = model.predict(images, verbose=0)
predicted = np.argmax(logits, axis=1)

plt.figure(figsize=(12, 12))
for i in range(min(25, len(images))):
    plt.subplot(5, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(images[i].numpy().astype("uint8"))

    true_name = class_names[int(labels[i])]
    predicted_name = class_names[int(predicted[i])]
    correct = true_name == predicted_name

    plt.xlabel(f"{true_name} / {predicted_name}",
               color="green" if correct else "red", fontsize=9)
plt.tight_layout()
plt.show()

print(f"Correct in this batch: {int((predicted == labels.numpy()).sum())} / {len(images)}")

---
# Chapter 2. Fine-tuning with data augmentation

Chapter 1 trained only the final layer. Chapter 2 unfreezes the convolutional base so its
pretrained filters can adjust to flowers specifically, and adds data augmentation -
random flips, rotations, shifts and brightness changes - so the model sees a slightly
different version of each photo every epoch.

**Two rules make this work, and breaking either one destroys the model:**

1. **The learning rate must be much smaller.** The pretrained weights are already good.
   Applying normal-sized gradient steps to them erases what they know before the new
   information arrives. The original divides its rate by 10; this notebook uses 1e-5,
   a hundredth of Adam's 1e-3 default, which is the usual recommendation for fine-tuning.
2. **BatchNorm layers must stay in inference mode.** EfficientNet is full of
   BatchNormalization layers holding running statistics learned from ImageNet. If they
   update on small flower batches, those statistics are overwritten with noisy estimates
   and accuracy collapses. Calling `base_model(inputs, training=False)` when the model was
   built - as Step 3 does - keeps them frozen even after `trainable = True`.

The original does not address the second point at all.

### What the original's own numbers show

This is worth being honest about, because it is the most interesting result in the
original notebook and it goes uncommented there. Its Chapter 2 output reads:

| Stage | Validation accuracy |
|---|---|
| After Chapter 1 (frozen, 3 epochs) | 0.8851 |
| After Chapter 2 (unfrozen + augmentation, 10 epochs) | 0.8759 |

**Ten further epochs made it slightly worse.** Training accuracy also fell, from 0.90 to
0.75, but that part is expected - augmented images are harder than clean ones, so a lower
training score does not itself indicate a problem. The validation score is the one that
matters, and it did not improve.

So fine-tuning is not automatically an improvement. It is a second tool with its own
failure modes, and whether it helps is an empirical question - which is why the
comparison below is built into this notebook rather than left to the reader.

In [ ]:
# CHANGE: .keras single file, and no unzip step.
# The original runs `!unzip saved_model_CIFAR10.zip`, which stops to ask
# "replace ...? [y]es, [n]o" when the folder already exists, hanging a fresh run.
model = tf.keras.models.load_model(export_path)
print("Reloaded from", export_path)

In [ ]:
# Find the pretrained base inside the reloaded model and unfreeze it.
base_model = next(
    layer for layer in model.layers
    if isinstance(layer, tf.keras.Model)
)

base_model.trainable = True

for layer in model.layers:
    print(f"Layer: {layer.name:28s} Trainable: {layer.trainable}")

### Setting the fine-tuning learning rate

The original mutates the optimizer in place with `optimizer.lr.assign(new_lr)`. `.lr` is
a deprecated alias for `.learning_rate` and in-place mutation leaves the optimizer's
internal state from the previous stage in place. Recompiling with a fresh optimizer is
clearer and resets that state.

In [ ]:
# CHANGE: recompile with a fresh low-rate optimizer instead of mutating optimizer.lr.
FINE_TUNE_LR = 1e-5

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

print("Fine-tuning learning rate:", FINE_TUNE_LR)

### Data augmentation

Two differences from the original, both deliberate.

**Where it is applied.** The original runs `data_augmentation(train_images)` on the
entire 50,000-image array in one call, once per epoch. That materialises a second full
copy of the dataset in memory, and every image is augmented exactly once per epoch.
Applying augmentation inside the `tf.data` pipeline instead augments each batch as it is
drawn, uses no extra memory, and overlaps with training.

**The brightness range.** The original passes `value_range=(0, 1)` because its pixels are
divided by 255. This pipeline keeps pixels in `[0, 255]`, so the range must match, or
`RandomBrightness` would shift every image to pure white.

In [ ]:
# CHANGE: augmentation applied per batch inside the pipeline, not to the whole
# array at once; value_range matches this notebook's [0, 255] pixels.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(factor=0.1, value_range=(0, 255)),
], name="data_augmentation")

augmented_train_ds = train_ds.map(
    lambda image, label: (data_augmentation(image, training=True), label),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)

# Only the training data is augmented. Validation and test stay clean, otherwise the
# scores would not be comparable with Chapter 1.
images, labels = next(iter(augmented_train_ds))

plt.figure(figsize=(10, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(np.clip(images[i].numpy(), 0, 255).astype("uint8"))
    plt.title(class_names[int(labels[i])], fontsize=10)
    plt.axis("off")
plt.suptitle("Augmented training images")
plt.tight_layout()
plt.show()

### Train the fine-tuning stage

The original wraps `model.fit(epochs=1)` in a Python `for` loop because it re-augments
the whole array between iterations. With augmentation inside the pipeline that is
unnecessary - a single `fit` call with several epochs does the same thing and keeps one
continuous `history` to plot.

(Incidentally, the original's loop reads `for i in range(3)` while printing
`Iteration {i+1}/10`, and its saved output shows ten iterations. The stored code and the
stored output do not come from the same run.)

In [ ]:
# CHANGE: one fit() call instead of a Python loop around fit(epochs=1).
fine_tune_history = model.fit(
    augmented_train_ds,
    validation_data=validation_ds,
    epochs=5,
)

## Did fine-tuning actually help?

The comparison the original leaves to the reader. Both stages are measured on the same
untouched test set.

In [ ]:
# CHANGE (new cell): state plainly whether stage 2 improved on stage 1.
fine_tuned_loss, fine_tuned_accuracy = model.evaluate(test_ds, verbose=0)

print(f"Chapter 1 (frozen base, 3 epochs):        {frozen_stage_accuracy:.4f}")
print(f"Chapter 2 (fine-tuned + augmentation):    {fine_tuned_accuracy:.4f}")

difference = fine_tuned_accuracy - frozen_stage_accuracy
print(f"Difference:                               {difference:+.4f}")
print()

if difference > 0.005:
    print("Fine-tuning helped.")
elif difference < -0.005:
    print("Fine-tuning made it WORSE - as happened in the original notebook.")
    print("Likely causes: learning rate still too high, too few epochs at this rate,")
    print("or augmentation too strong for a dataset this small.")
else:
    print("No meaningful change. The frozen features were already close to")
    print("the best this architecture gets on this dataset.")

In [ ]:
# CHANGE (new cell): both stages' curves on one axis.
plt.figure(figsize=(7, 4.5))

stage1 = history.history["val_accuracy"]
stage2 = fine_tune_history.history["val_accuracy"]

plt.plot(range(1, len(stage1) + 1), stage1, marker="o", label="Chapter 1: frozen base")
plt.plot(range(len(stage1) + 1, len(stage1) + len(stage2) + 1), stage2,
         marker="o", label="Chapter 2: fine-tuned + augmentation")

plt.axvline(len(stage1) + 0.5, linestyle="--", linewidth=1, color="grey")
plt.xlabel("epoch"); plt.ylabel("validation accuracy")
plt.title("Validation accuracy across both stages")
plt.legend()
plt.tight_layout()
plt.show()

### Where the remaining mistakes are

Accuracy says how many are wrong; a confusion matrix says **which** are wrong. For flower
species that is the useful question, because some pairs genuinely look alike.

In [ ]:
# CHANGE (new cell): confusion matrix over the whole test set.
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

y_true, y_pred = [], []
for images, labels in test_ds:
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(model.predict(images, verbose=0), axis=1))

y_true, y_pred = np.array(y_true), np.array(y_pred)

print(pd.DataFrame(
    confusion_matrix(y_true, y_pred),
    index=[f"actual: {n}" for n in class_names],
    columns=[f"pred: {n}" for n in class_names],
))
print()
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

---
# Summary

## The pipeline, unchanged from the original

1. Load a labelled image dataset and split it into training, validation and test sets.
2. Resize every image to the resolution the pretrained network expects.
3. Load a convolutional network pretrained on ImageNet, **without** its classifier head,
   and freeze it.
4. Add one `Dense(num_classes)` layer producing logits.
5. Train that layer only, then optionally unfreeze and fine-tune at a much lower rate.
6. Evaluate on data the model has never seen.

## What changed, and why

| # | Change | Reason |
|---|---|---|
| 1 | tf_flowers instead of CIFAR-10 | real photographs at natural resolution suit a pretrained photo model; CIFAR-10's 32x32 images upscaled to 240x240 waste the pretrained features |
| 2 | Manual 80/10/10 slicing | tf_flowers ships only a `train` split; CIFAR-10 ships train and test |
| 3 | `class_names` read from dataset metadata | the original hard-codes the list, which breaks silently if label order changes |
| 4 | `tf.data` pipeline instead of NumPy arrays | tf_flowers images vary in size and cannot be stacked before resizing; also avoids holding the dataset twice in memory |
| 5 | **No division by 255** | `keras.applications` EfficientNetV2 rescales internally and expects `[0, 255]`; TF-Hub modules expect `[0, 1]`. Getting this wrong degrades accuracy without any error |
| 6 | `keras.applications.EfficientNetV2B1` instead of `hub.KerasLayer` + `tf_keras` | removes the Keras 2 compatibility shim the original needs; same architecture and weights |
| 7 | `base_model(inputs, training=False)` | keeps BatchNorm statistics frozen during fine-tuning - the original never addresses this |
| 8 | Recompile with `Adam(1e-5)` instead of `optimizer.lr.assign(...)` | `.lr` is deprecated, and recompiling resets stale optimizer state |
| 9 | Augmentation inside the pipeline | the original augments the entire array in one call, doubling memory use |
| 10 | `value_range=(0, 255)` in `RandomBrightness` | must match this pipeline's pixel range |
| 11 | One `fit(epochs=5)` instead of a loop around `fit(epochs=1)` | unnecessary once augmentation is in the pipeline; keeps one continuous history |
| 12 | `.keras` single file instead of SavedModel + zip | removes the `unzip` step, which prompts interactively and hangs a fresh run |
| 13 | Batched prediction for the image grid | the original predicts one image at a time in a Python loop |
| 14 | Explicit stage-1 vs stage-2 comparison, confusion matrix, learning curves | the original's own output shows fine-tuning made things slightly worse and never says so |

## Limitations

- 3,670 images across 5 classes is a small dataset. The test split is roughly 367 images,
  so test accuracy carries a sampling uncertainty of a couple of percentage points, and
  small differences between the two stages should not be over-read.
- Only the classifier head is trained in Chapter 1, so the model cannot learn features
  ImageNet does not already contain. For subjects far from ImageNet's domain - medical
  scans, industrial defects - freezing the base is a much weaker starting point.
- Fine-tuning is sensitive to learning rate and epoch count, neither of which is tuned
  here. A negative result from one configuration is not proof that fine-tuning cannot help.
- The flower classes are visually distinct. Accuracy on fine-grained tasks, where classes
  differ only in small details, would be considerably lower.